## Test automated Agent's LLM model inference (infer call)

We'll create an Agent that has an LLM. Protolink offers automated LLM invocation, with builtin tool calling and reasoning capabilities. You as a user can delegate LLM calls to the Agent, and it will handle the invocation automatically.
Also in the Agent's constructor you can pass a system prompt, which will be used to guide the LLM's behavior.

Args:
- `system_prompt`: This is used as complementary text in the system prompt, which is responsible for explaining the agent logic and role. The agent calling, tool calling and other A2A functionalities are already predefined, so the LLM already has the knowledge on how to interact with its environment. If you wish to override the system prompt completely, set `override_system_prompt` to True
- `override_system_prompt`: If True, overrides `system_prompt` completely with the `system_prompt` provided

The following cell will keep on appending the Agent's logs to the output cell.

In [ ]:
from protolink.agents import Agent
from protolink.llms.api import OpenAILLM
from protolink.llms.server import OllamaLLM  # noqa: F401

API_KEY = ""  # Enter your Key here

AGENT_URL = "http://localhost:8050"

llm = OpenAILLM(api_key=API_KEY, model="gpt-4o")

# Deploy Ollama locally so you can test for free!
# llm = OllamaLLM(base_url="http://localhost:11434")

# Here we'll pass directly a dict in the card argument instead of defining manually an AgentCard var. The agent will take care of it  # noqa: E501
# Pass a simple system prompt. The prompts for tool calling, agent calling are already predefined in the llms/prompts
agent = Agent(
    card={
        "name": "Test Agent",
        "description": "A test agent for LLM inference",
        "url": AGENT_URL,
        "author": "Test User",
    },
    transport="http",
    llm=llm,
    system_prompt="You are a helpful assistant that can answer questions.",
)

await agent.start()

## Task Creation

The Agents exchange Tasks between them. A `Task` is passed back and forth between agents to coordinate work. Each agent appends a `Message` or an `Artifact` to the task.
Each Message or Artifact contains one or more `Part` objects, which are the actual data being exchanged.
In this example we'll create a Task that contains a Message with an `infer part`. **The `infer Part` is sent when we want to invoke the other agent's LLM.**

In [ ]:
from protolink.models import Message, Part, Task

task = Task(
    messages=[Message(role="user", parts=[Part(type="infer", content={"prompt": "What is the capital of Greece?"})])]
)
print(task.to_dict())
# or use the builtin convenience method for the same task
task = Task.create_infer(prompt="What is the capital of Greece?")
print(task.to_dict())

## Agent Client

The way to communicate with an agent is through the **Agent Client**. It provides a simple interface to **send tasks** to an agent and **receive responses**.
One way would be to create an agent whose purpose is just to use for communication with other agents.
We'll do something simpler here and initiate an Agent Client directly and use it's `send_task` method.

In [ ]:
from protolink.client import AgentClient

# setup client using the Agent's Transport
client = AgentClient(transport=agent.transport)

# Send the Task to the Agent
result = await client.send_task(agent_url=AGENT_URL, task=task)
# use the task's builtin helper function to get the content of the last part
print(result.get_last_part_content())

## Tool invocation

Let's try invoking a tool. We'll ask first a question that requires real-time data where the LLM will fail to answer. Then we'll provide a tool to the Agent that responds to this question and see the LLM use it. The user has to do Nothing, protolink handles the tool calling from the LLM automatically.



In [ ]:
task = Task(messages=[Message(role="user", parts=[Part.infer(prompt="What's the weather right now in Geneva?")])])

print(f"Task:\n{task}")
result = await client.send_task(agent_url=AGENT_URL, task=task)
print(f"Response:\n{result.get_last_part_content()}")

##### Let's **stop** the Agent and **add a tool** that fetches the Weather. and see if not the LLM will reply with the weather is sunny by invoking this tool.

In [ ]:
await agent.stop()


@agent.tool(name="weather_info", description="Get weather information for a location", input_schema={"location": "str"})
def get_weather(location: str) -> str:
    return f"The weather in {location} is sunny."


# Start again
await agent.start()

In [ ]:
# Ask the question again
result = await client.send_task(agent_url=AGENT_URL, task=task)
print(result.get_last_part_content())